# Job 1: Embeddings & FAISS Index

Generates content-based embeddings for 748K movie items and builds FAISS similarity index.

**Resumable:** Re-running skips completed stages via checkpoints.

In [0]:
%pip install --upgrade numpy==1.26.4 sentence-transformers faiss-cpu tqdm requests
dbutils.library.restartPython()

In [0]:
import json
import os
import sys
import time
import logging

import numpy as np
import pandas as pd
import requests

# Load TMDB API key from secrets
_secrets = json.loads(
    dbutils.fs.head("dbfs:/Workspace/Users/mqwebster238@gmail.com/secrets.json")
)
TMDB_API_KEY = _secrets["TMDB_API_KEY"]

# Add custom modules to path
sys.path.append('/Workspace/Users/mqwebster238@gmail.com/novametrics/src/')

from features import build_embedding_input, get_embedding_tier
from model_cb import build_faiss_index, save_index, load_index, query_index

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
log = logging.getLogger(__name__)

In [0]:
# All paths and parameters
OUTPUTS_DIR       = "/Volumes/movie_recsys/data/outputs"
META_CLEAN_PATH   = f"{OUTPUTS_DIR}/meta_clean.parquet"
REVIEWS_PATH      = f"{OUTPUTS_DIR}/reviews_5core.parquet"
MOST_HELPFUL_PATH = f"{OUTPUTS_DIR}/most_helpful.parquet"
TMDB_CHECKPOINT   = f"{OUTPUTS_DIR}/tmdb_enriched.parquet"
EMBEDDINGS_PATH   = f"{OUTPUTS_DIR}/embeddings.npy"
ASIN_INDEX_PATH   = f"{OUTPUTS_DIR}/asin_index.npy"
FAISS_INDEX_PATH  = f"{OUTPUTS_DIR}/faiss_index.bin"

EMBEDDING_MODEL   = "all-MiniLM-L6-v2"
EMBEDDING_DIM     = 384
BATCH_SIZE        = 256        # REDUCED from 512 for memory efficiency
N_CLUSTERS        = 256        # CONFIG PARAM — IVF cells for FAISS
MAX_REVIEW_WORDS  = 256        # CONFIG PARAM — word cap on review text
CHECKPOINT_EVERY  = 100        # INCREASED from 50 - save more frequently
LOG_EVERY         = 10         # progress log every N batches

# Fine-grained regeneration flags — set True to bypass that stage's checkpoint.
# FORCE_TMDB:       re-runs the ~134-min TMDB API loop from scratch.
# FORCE_EMBEDDINGS: re-runs the ~5-hr embedding loop from scratch.
# Leave both False for normal resumable runs.
FORCE_TMDB        = False
FORCE_EMBEDDINGS  = False

TMDB_SEARCH_URL   = "https://api.themoviedb.org/3/search/movie"
TMDB_SLEEP        = 1.0 / 40  # 40 req/s free-tier rate limit

SPOT_CHECK_TITLES = ["The Dark Knight", "Toy Story", "The Godfather"]
SPOT_CHECK_K      = 5

In [0]:
_STR_COLS = [
    "title", "genres_str", "description_str", "most_helpful_review",
    "tmdb_title", "tmdb_description", "tmdb_genres",
    "title_final", "genres_final", "description_final",
]

def _clean_str_cols(df: pd.DataFrame, cols: list = _STR_COLS) -> pd.DataFrame:
    """
    Coerce string columns to clean str | None in-place.
    Handles float NaN, string 'nan', empty strings, and non-object dtypes.
    """
    for col in cols:
        if col not in df.columns:
            continue
        if df[col].dtype != object:
            log.warning("Column '%s' has dtype %s, expected str/object — coercing.", col, df[col].dtype)
        df[col] = df[col].apply(
            lambda v: None if (
                v is None
                or (isinstance(v, float) and pd.isna(v))
                or str(v).strip().lower() == "nan"
                or str(v).strip() == ""
            ) else str(v).strip()
        )
    return df

def _is_present(v) -> bool:
    """Return True only for non-empty, non-NaN string values."""
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return False
    s = str(v).strip()
    return bool(s) and s.lower() != "nan"

def _coalesce(primary, fallback):
    """Return the first present value as a clean string, or None."""
    if _is_present(primary):
        return str(primary).strip()
    if _is_present(fallback):
        return str(fallback).strip()
    return None

In [0]:
log.info("Loading meta_clean from %s", META_CLEAN_PATH)
meta = pd.read_parquet(META_CLEAN_PATH)
assert "parent_asin" in meta.columns, "Expected parent_asin in meta_clean.parquet"
_clean_str_cols(meta)
log.info("meta_clean: %d rows | dtypes checked and string cols normalised", len(meta))

if os.path.exists(MOST_HELPFUL_PATH):
    log.info("most_helpful checkpoint found — skipping reviews load.")
    most_helpful = pd.read_parquet(MOST_HELPFUL_PATH)
else:
    log.info("Loading reviews from %s", REVIEWS_PATH)
    reviews = pd.read_parquet(REVIEWS_PATH, columns=["parent_asin", "helpful_vote", "text"])
    log.info("reviews_5core: %d rows — building most_helpful …", len(reviews))
    most_helpful = (
        reviews
        .sort_values("helpful_vote", ascending=False)
        .groupby("parent_asin", as_index=False)
        .first()[["parent_asin", "text"]]
        .rename(columns={"text": "most_helpful_review"})
    )
    del reviews
    os.makedirs(OUTPUTS_DIR, exist_ok=True)
    most_helpful.to_parquet(MOST_HELPFUL_PATH, index=False)
    log.info("most_helpful saved to %s", MOST_HELPFUL_PATH)

_clean_str_cols(most_helpful, ["most_helpful_review"])
log.info("most_helpful: %d items, %d with review text",
         len(most_helpful), most_helpful["most_helpful_review"].notna().sum())

# Join → meta_with_review (748,224 rows, left join keeps all meta items)
meta = meta.merge(most_helpful, on="parent_asin", how="left")
_clean_str_cols(meta)
log.info("meta_with_review: %d rows | string cols normalised after merge", len(meta))

In [0]:
def _fetch_tmdb(title: str, api_key: str, session: requests.Session) -> dict | None:
    try:
        resp = session.get(
            TMDB_SEARCH_URL,
            params={"api_key": api_key, "query": title, "language": "en-US", "page": 1},
            timeout=10,
        )
        resp.raise_for_status()
        results = resp.json().get("results", [])
        if not results:
            return None
        top = results[0]
        genre_str = "|".join(str(g) for g in top.get("genre_ids", []))
        return {
            "title":       top.get("title", ""),
            "description": top.get("overview", ""),
            "genres":      genre_str,
        }
    except Exception as exc:
        log.warning("TMDB fetch failed for title '%s': %s", title, exc)
        return None

def run_tmdb_enrichment(tier4_df: pd.DataFrame, api_key: str) -> pd.DataFrame:
    records = []
    session = requests.Session()
    total   = len(tier4_df)

    for i, (_, row) in enumerate(tier4_df.iterrows()):
        if i % 500 == 0:
            log.info("TMDB enrichment: %d / %d (%.1f%%)", i, total, 100 * i / max(total, 1))
        result = _fetch_tmdb(row["title"] or "", api_key, session)
        records.append({
            "parent_asin":      row["parent_asin"],
            "tmdb_title":       result["title"]       if result else None,
            "tmdb_description": result["description"] if result else None,
            "tmdb_genres":      result["genres"]      if result else None,
        })
        time.sleep(TMDB_SLEEP)

    session.close()
    return pd.DataFrame(records)

In [0]:
# Identify Tier 4 before any enrichment
meta["_emb_input"] = meta.apply(
    lambda r: build_embedding_input(
        r["title"], r["genres_str"], r["description_str"], r["most_helpful_review"],
        max_review_words=MAX_REVIEW_WORDS,
    ),
    axis=1,
)
tier4_mask = meta["_emb_input"].isna()
tier4_df   = meta[tier4_mask].copy()
log.info("Tier 4 items (need TMDB): %d / %d (%.1f%%)",
         len(tier4_df), len(meta), 100 * len(tier4_df) / len(meta))

if not FORCE_TMDB and os.path.exists(TMDB_CHECKPOINT):
    log.info("TMDB checkpoint found — skipping API loop.")
    tmdb_enriched = pd.read_parquet(TMDB_CHECKPOINT)
else:
    if not TMDB_API_KEY:
        raise EnvironmentError(
            "TMDB_API_KEY not found in secrets.json. "
            "Add it at dbfs:/Workspace/Users/mqwebsters238@gmail.com/secrets.json"
        )
    log.info("Starting TMDB enrichment for %d items …", len(tier4_df))
    tmdb_enriched = run_tmdb_enrichment(tier4_df, TMDB_API_KEY)
    os.makedirs(OUTPUTS_DIR, exist_ok=True)
    tmdb_enriched.to_parquet(TMDB_CHECKPOINT, index=False)
    log.info("TMDB checkpoint saved to %s", TMDB_CHECKPOINT)

_clean_str_cols(tmdb_enriched, ["tmdb_title", "tmdb_description", "tmdb_genres"])

In [0]:
meta = meta.merge(tmdb_enriched, on="parent_asin", how="left")
_clean_str_cols(meta)   # normalise tmdb_* cols before coalescing

meta["title_final"]       = meta.apply(lambda r: _coalesce(r["title"],           r.get("tmdb_title")),       axis=1)
meta["genres_final"]      = meta.apply(lambda r: _coalesce(r["genres_str"],      r.get("tmdb_genres")),      axis=1)
meta["description_final"] = meta.apply(lambda r: _coalesce(r["description_str"], r.get("tmdb_description")), axis=1)

# Final normalisation pass on the coalesced columns
_clean_str_cols(meta, ["title_final", "genres_final", "description_final"])

log.info(
    "Post-TMDB coverage — title: %.1f%%, genres: %.1f%%, description: %.1f%%",
    meta["title_final"].notna().mean() * 100,
    meta["genres_final"].notna().mean() * 100,
    meta["description_final"].notna().mean() * 100,
)
meta.drop(columns=["_emb_input"], inplace=True, errors="ignore")


In [0]:
meta["embedding_input"] = meta.apply(
    lambda r: build_embedding_input(
        r["title_final"], r["genres_final"],
        r["description_final"], r["most_helpful_review"],
        max_review_words=MAX_REVIEW_WORDS,
    ),
    axis=1,
)
meta["embedding_tier"] = meta.apply(
    lambda r: get_embedding_tier(
        r["title_final"], r["genres_final"],
        r["description_final"], r["most_helpful_review"],
    ),
    axis=1,
)

tier_counts = meta["embedding_tier"].value_counts().sort_index()
log.info("Tier distribution after TMDB enrichment:")
for tier, count in tier_counts.items():
    log.info("  Tier %d: %6d items (%5.1f%%)", tier, count, 100 * count / len(meta))

true_gaps  = meta["embedding_input"].isna()
embeddable = meta[~true_gaps].reset_index(drop=True)
log.info("True gaps after TMDB: %d — skipped. Items to embed: %d", true_gaps.sum(), len(embeddable))

In [0]:
# STEP A: Diagnose current state (handling numpy compatibility)
import sys
sys.path.append('/Workspace/Users/mqwebster238@gmail.com/novametrics/src/')
from features import build_embedding_input
import pandas as pd
import numpy as np
import os

OUTPUTS_DIR = "/Volumes/movie_recsys/data/outputs"

# Check current checkpoint state
emb_path = f"{OUTPUTS_DIR}/embeddings.npy"
asin_path = f"{OUTPUTS_DIR}/asin_index.npy"
faiss_path = f"{OUTPUTS_DIR}/faiss_index.bin"

print("Checking current checkpoint files...")
if os.path.exists(emb_path):
    emb_current = np.load(emb_path)
    print(f"✓ embeddings.npy exists: {len(emb_current):,} items, shape {emb_current.shape}")
    print(f"  Batch calculation: {len(emb_current):,} / 512 = {len(emb_current) / 512:.1f} batches")
    print(f"  256,000 / 512 = {256000 / 512:.0f} batches exactly ← suspect this is the cap")
    
    # Try to load asin_index, but handle numpy compatibility issues
    try:
        asin_current = np.load(asin_path, allow_pickle=True)
        print(f"✓ asin_index.npy exists: {len(asin_current):,} items")
    except Exception as e:
        print(f"⚠ asin_index.npy exists but has compatibility issue: {type(e).__name__}")
        print(f"  (This is OK - we'll regenerate it with the embeddings)")
else:
    print("✗ No checkpoint found")

if os.path.exists(faiss_path):
    import struct
    with open(faiss_path, 'rb') as f:
        f.seek(0)
        data = f.read(100)
    print(f"✓ faiss_index.bin exists ({os.path.getsize(faiss_path) / (1024**2):.1f} MB)")
else:
    print("✗ faiss_index.bin not found")

# Reconstruct embeddable dataset to see what SHOULD be embedded
print("\nReconstructing embeddable dataset from source files...")
meta = pd.read_parquet(f"{OUTPUTS_DIR}/meta_clean.parquet")
most_helpful = pd.read_parquet(f"{OUTPUTS_DIR}/most_helpful.parquet")
tmdb = pd.read_parquet(f"{OUTPUTS_DIR}/tmdb_enriched.parquet")

meta = meta.merge(most_helpful, on="parent_asin", how="left")
meta = meta.merge(tmdb, on="parent_asin", how="left")

# String cleaning function
def _is_present(v):
    if v is None: return False
    import math
    if isinstance(v, float) and math.isnan(v): return False
    return bool(str(v).strip()) and str(v).strip().lower() != "nan"

def _coalesce(a, b):
    return str(a).strip() if _is_present(a) else (str(b).strip() if _is_present(b) else None)

meta["title_final"] = meta.apply(lambda r: _coalesce(r.get("title"), r.get("tmdb_title")), axis=1)
meta["genres_final"] = meta.apply(lambda r: _coalesce(r.get("genres_str"), r.get("tmdb_genres")), axis=1)
meta["description_final"] = meta.apply(lambda r: _coalesce(r.get("description_str"), r.get("tmdb_description")), axis=1)

meta["embedding_input"] = meta.apply(
    lambda r: build_embedding_input(
        r["title_final"], r["genres_final"],
        r["description_final"], r.get("most_helpful_review"),
        max_review_words=256
    ), axis=1
)

true_gaps = meta["embedding_input"].isna()
embeddable = meta[~true_gaps]

print(f"\nDataset composition:")
print(f"  Total meta items:          {len(meta):,}")
print(f"  Embeddable items (target): {len(embeddable):,}")
print(f"  True gaps (unembeddable):  {true_gaps.sum():,}")

if os.path.exists(emb_path):
    missing = len(embeddable) - len(emb_current)
    completion_pct = len(emb_current) / len(embeddable) * 100
    
    print(f"\n{'='*70}")
    print(f"DIAGNOSIS RESULT:")
    print(f"{'='*70}")
    print(f"Expected embeddable items : {len(embeddable):,}")
    print(f"Actually embedded items   : {len(emb_current):,} ({completion_pct:.1f}% complete)")
    print(f"MISSING from checkpoint   : {missing:,}")
    print(f"\nLoop status:")
    print(f"  Stopped at batch {len(emb_current) // 512} of ~{len(embeddable) // 512} needed")
    print(f"  Resume point: batch {len(emb_current) // 512}, item {len(emb_current):,}")
    print(f"  Remaining batches: {(len(embeddable) - len(emb_current)) // 512 + 1}")
    print(f"  ETA: ~{((len(embeddable) - len(emb_current)) / 512 * 10):.0f}-{((len(embeddable) - len(emb_current)) / 512 * 20):.0f} minutes")
    print(f"{'='*70}")
    
    # Store embeddable for use in next cells
    print(f"\n✓ embeddable dataframe ready for Step 5 ({len(embeddable):,} items)")

In [0]:
# STEP B: Memory-Efficient Chunked Resume (Fixed)
from sentence_transformers import SentenceTransformer
import time
import gc

log.info("="*70)
log.info("CHUNKED EMBEDDING RESUME (Memory-Efficient)")
log.info("="*70)

# Check current checkpoint state
_ckpt_emb = np.load(EMBEDDINGS_PATH)
_ckpt_n = len(_ckpt_emb)
del _ckpt_emb
gc.collect()

log.info("Current checkpoint: %d items", _ckpt_n)

if _ckpt_n >= 433000:
    log.info("✅ Checkpoint complete (%d items) - skip to FAISS rebuild", _ckpt_n)
    print(f"\n✅ Embeddings already complete: {_ckpt_n:,} items")
else:
    # Build embeddable dataset ONCE (not per chunk)
    log.info("Building embeddable dataset...")
    meta = pd.read_parquet(META_CLEAN_PATH)
    most_helpful = pd.read_parquet(MOST_HELPFUL_PATH)
    tmdb = pd.read_parquet(TMDB_CHECKPOINT)
    
    meta = meta.merge(most_helpful, on="parent_asin", how="left")
    meta = meta.merge(tmdb, on="parent_asin", how="left")
    
    def _coalesce(a, b):
        if a and str(a).strip() and str(a).strip().lower() != "nan": return str(a).strip()
        if b and str(b).strip() and str(b).strip().lower() != "nan": return str(b).strip()
        return None
    
    meta["title_final"] = meta.apply(lambda r: _coalesce(r.get("title"), r.get("tmdb_title")), axis=1)
    meta["genres_final"] = meta.apply(lambda r: _coalesce(r.get("genres_str"), r.get("tmdb_genres")), axis=1)
    meta["description_final"] = meta.apply(lambda r: _coalesce(r.get("description_str"), r.get("tmdb_description")), axis=1)
    
    meta["embedding_input"] = meta.apply(
        lambda r: build_embedding_input(
            r["title_final"], r["genres_final"],
            r["description_final"], r.get("most_helpful_review"),
            max_review_words=MAX_REVIEW_WORDS
        ), axis=1
    )
    
    embeddable = meta[meta["embedding_input"].notna()].reset_index(drop=True)
    texts_all = embeddable["embedding_input"].tolist()
    asins_all = embeddable["parent_asin"].tolist()
    n_total = len(texts_all)
    
    log.info("Embeddable dataset ready: %d items total", n_total)
    
    # Free the large dataframes
    del meta, most_helpful, tmdb, embeddable
    gc.collect()
    
    # Slice to get only remaining items
    texts_remaining = texts_all[_ckpt_n:]
    asins_remaining = asins_all[_ckpt_n:]
    n_remaining = len(texts_remaining)
    
    log.info("Already embedded: %d items", _ckpt_n)
    log.info("Remaining to process: %d items", n_remaining)
    
    # Chunked processing on remaining items only
    CHUNK_SIZE = 25600
    n_chunks = (n_remaining + CHUNK_SIZE - 1) // CHUNK_SIZE
    
    log.info("Chunked plan: %d chunks of %d items", n_chunks, CHUNK_SIZE)
    log.info("Batch size: %d", BATCH_SIZE)
    
    # Load model
    log.info("Loading model: %s", EMBEDDING_MODEL)
    model = SentenceTransformer(EMBEDDING_MODEL)
    log.info("Model loaded")
    
    # Process each chunk from remaining items
    for chunk_idx in range(n_chunks):
        chunk_lo = chunk_idx * CHUNK_SIZE
        chunk_hi = min(chunk_lo + CHUNK_SIZE, n_remaining)
        chunk_actual = chunk_hi - chunk_lo
        
        log.info("="*70)
        log.info("CHUNK %d/%d: processing %d items (global position %d-%d)",
                 chunk_idx + 1, n_chunks, chunk_actual,
                 _ckpt_n + chunk_lo, _ckpt_n + chunk_hi - 1)
        log.info("="*70)
        
        # Extract this chunk from remaining slices
        texts_chunk = texts_remaining[chunk_lo:chunk_hi]
        asins_chunk = asins_remaining[chunk_lo:chunk_hi]
        
        log.info("Encoding %d texts...", len(texts_chunk))
        
        # Encode chunk in batches
        chunk_embeddings = np.zeros((len(texts_chunk), EMBEDDING_DIM), dtype=np.float32)
        n_batches = (len(texts_chunk) + BATCH_SIZE - 1) // BATCH_SIZE
        
        t_start = time.time()
        for batch_idx in range(n_batches):
            batch_lo = batch_idx * BATCH_SIZE
            batch_hi = min(batch_lo + BATCH_SIZE, len(texts_chunk))
            
            chunk_embeddings[batch_lo:batch_hi] = model.encode(
                texts_chunk[batch_lo:batch_hi],
                batch_size=BATCH_SIZE,
                show_progress_bar=False,
                convert_to_numpy=True,
                normalize_embeddings=False,
            ).astype(np.float32)
            
            if (batch_idx + 1) % 20 == 0 or batch_idx == n_batches - 1:
                elapsed = time.time() - t_start
                done = batch_hi
                rate = done / elapsed if elapsed > 0 else 0
                eta = (len(texts_chunk) - done) / rate if rate > 0 else 0
                log.info("  Batch %d/%d | %d/%d | %.0f items/s | ETA %.0fs",
                         batch_idx + 1, n_batches, done, len(texts_chunk), rate, eta)
        
        elapsed = time.time() - t_start
        log.info("Chunk encoded in %.1f s (%.0f items/s)",
                 elapsed, len(texts_chunk) / elapsed)
        
        # Append to checkpoint
        log.info("Appending to checkpoint...")
        existing_emb = np.load(EMBEDDINGS_PATH)
        existing_asin = np.load(ASIN_INDEX_PATH, allow_pickle=True)
        
        new_emb = np.vstack([existing_emb, chunk_embeddings])
        new_asin = np.concatenate([existing_asin, np.array(asins_chunk, dtype=object)])
        
        np.save(EMBEDDINGS_PATH, new_emb)
        np.save(ASIN_INDEX_PATH, new_asin)
        
        new_count = len(new_emb)
        progress_pct = new_count / n_total * 100
        log.info("✓ Checkpoint: %d total items (%.1f%% complete)",
                 new_count, progress_pct)
        
        del chunk_embeddings, existing_emb, existing_asin, new_emb, new_asin
        gc.collect()
        
        log.info("Chunk %d/%d complete. Memory freed.", chunk_idx + 1, n_chunks)
    
    log.info("="*70)
    log.info("✅ ALL CHUNKS COMPLETE")
    log.info("="*70)
    final_emb = np.load(EMBEDDINGS_PATH)
    log.info("Final: %d items embedded", len(final_emb))
    del final_emb
    
    print(f"\n✅ Embedding generation complete: {n_total:,} items")
    print(f"   Checkpoint: {EMBEDDINGS_PATH}")

In [0]:
# STEP C: Rebuild FAISS index from the complete embedding set
import gc

log.info("="*70)
log.info("REBUILDING FAISS INDEX")
log.info("="*70)

# Load final embeddings
log.info("Loading complete embeddings from %s", EMBEDDINGS_PATH)
all_embeddings = np.load(EMBEDDINGS_PATH)
all_asins = np.load(ASIN_INDEX_PATH, allow_pickle=True)

log.info("Loaded embeddings: shape=%s", all_embeddings.shape)
log.info("Loaded ASINs: %d items", len(all_asins))

# Validate
assert len(all_embeddings) == len(all_asins), f"Mismatch: {len(all_embeddings)} embeddings vs {len(all_asins)} ASINs"
assert all_embeddings.shape[1] == EMBEDDING_DIM, f"Wrong dimension: {all_embeddings.shape[1]} vs expected {EMBEDDING_DIM}"
assert len(all_embeddings) >= 400000, f"Too few embeddings: {len(all_embeddings):,} (expected > 400K)"

log.info("✓ Validation passed")

# Build FAISS index
n_clusters_actual = min(N_CLUSTERS, len(all_embeddings))
if n_clusters_actual < N_CLUSTERS:
    log.warning("Reducing n_clusters from %d to %d", N_CLUSTERS, n_clusters_actual)

log.info("Building FAISS IVF-Flat index with %d clusters...", n_clusters_actual)
index = build_faiss_index(all_embeddings, n_clusters=n_clusters_actual)

log.info("Saving index to %s", FAISS_INDEX_PATH)
save_index(index, FAISS_INDEX_PATH)

log.info("="*70)
log.info("✅ FAISS INDEX COMPLETE")
log.info("="*70)
log.info("  Index ntotal     : %d", index.ntotal)
log.info("  Embeddings count : %d", len(all_embeddings))
log.info("  Match check      : %s", "✓ PASS" if index.ntotal == len(all_embeddings) else "✗ FAIL")
log.info("  Index file       : %s", FAISS_INDEX_PATH)
log.info("  Index size       : %.1f MB", os.path.getsize(FAISS_INDEX_PATH) / (1024**2))
log.info("="*70)

print(f"\n✅ FAISS index rebuilt: {index.ntotal:,} vectors")

# Spot check
log.info("\nSpot check: query with first embedding...")
dists, idxs = query_index(index, all_embeddings[0:1], k=6)
log.info("  Returned %d results", len(idxs[0]))
log.info("  Rank-0 distance: %.6f (should be 0.0 for self)", dists[0][0])
log.info("  Rank-1 distance: %.6f (should be > 0)", dists[0][1])

if dists[0][0] < 0.001 and dists[0][1] > 0:
    log.info("✓ Spot check PASSED")
else:
    log.warning("⚠ Spot check questionable - review results")

print("\n✓ Ready for validation script")

In [0]:
# STEP D: Complete validation
import sys
sys.path.append('/Workspace/Users/mqwebster238@gmail.com/novametrics/src/')
from model_cb import load_index, query_index

OUTPUTS_DIR = "/Volumes/movie_recsys/data/outputs"
embeddings = np.load(f"{OUTPUTS_DIR}/embeddings.npy")
all_asins = np.load(f"{OUTPUTS_DIR}/asin_index.npy", allow_pickle=True)
index = load_index(f"{OUTPUTS_DIR}/faiss_index.bin")
meta = pd.read_parquet(f"{OUTPUTS_DIR}/meta_clean.parquet")

results = {}
def check(name, passed, detail=""):
    tag = "✅" if passed else "❌"
    results[name] = passed
    print(f"  {tag}  {name}")
    if detail: print(f"       {detail}")

print("="*65)
print("JOB 1 FINAL VALIDATION")
print("="*65)

n_emb = len(embeddings)
pct = n_emb / len(meta) * 100

print("\nT1: Completeness")
check("Has > 400K embeddings", n_emb > 400000, f"{n_emb:,}")
check("Coverage >= 55%", pct >= 55, f"{pct:.1f}%")

print("\nT2: Shape")
check("Dim = 384", embeddings.shape[1] == 384)
check("ASIN count matches", len(all_asins) == n_emb)
check("FAISS ntotal matches", index.ntotal == n_emb)

print("\nT3: Quality")
has_nan = np.isnan(embeddings).any()
n_zero = (np.abs(embeddings).sum(axis=1) == 0).sum()
norms = np.linalg.norm(embeddings, axis=1)
check("No NaN", not has_nan)
check("No zeros", n_zero == 0)
check("Norm ~ 1.0", 0.95 <= norms.mean() <= 1.05, f"{norms.mean():.3f}")

print("\nT4: FAISS")
dists, idxs = query_index(index, embeddings[0:1], k=6)
check("Returns 6", dists.shape == (1, 6))
check("Self-query works", idxs[0][0] == 0 and dists[0][0] < 0.001)

print("\nT5: ASINs")
n_null = sum(1 for a in all_asins if not a or str(a).strip() == "")
overlap = len(set(str(a) for a in all_asins) & set(meta["parent_asin"].astype(str)))
check("No nulls", n_null == 0)
check("95% overlap", overlap / len(all_asins) >= 0.95, f"{overlap/len(all_asins)*100:.1f}%")

print("\n" + "="*65)
p = sum(results.values())
f = len(results) - p
print(f"RESULT: {p} passed, {f} failed")
if f == 0:
    print(f"✅ JOB 1 COMPLETE: {n_emb:,} items ({pct:.1f}% coverage)")
    print("Ready for Job 2")
else:
    print(f"❌ {f} check(s) failed - review errors above")
print("="*65)

In [0]:
n_clusters_actual = min(N_CLUSTERS, len(all_embeddings))
if n_clusters_actual < N_CLUSTERS:
    log.warning("Reducing n_clusters from %d to %d to match embedding count",
                N_CLUSTERS, n_clusters_actual)
log.info("Building FAISS IVF-Flat index: n_items=%d, dim=%d, n_clusters=%d",
         len(all_embeddings), EMBEDDING_DIM, n_clusters_actual)

index = build_faiss_index(all_embeddings, n_clusters=n_clusters_actual)
save_index(index, FAISS_INDEX_PATH)
log.info("FAISS index saved to %s | ntotal=%d", FAISS_INDEX_PATH, index.ntotal)

In [0]:
index = load_index(FAISS_INDEX_PATH)
assert index.ntotal == len(all_embeddings), (
    f"Index ntotal ({index.ntotal}) != embeddings ({len(all_embeddings)})"
)
log.info("Index integrity check passed: ntotal=%d", index.ntotal)

asin_to_idx   = {asin: i for i, asin in enumerate(all_asins)}
asin_to_title = meta.set_index("parent_asin")["title_final"].fillna("(unknown)").to_dict()

log.info("Spot-check: top-%d neighbours", SPOT_CHECK_K)
for seed_title in SPOT_CHECK_TITLES:
    matches = meta[meta["title_final"].str.contains(seed_title, case=False, na=False)]
    if matches.empty:
        log.info("  '%s': not found in metadata — skipping", seed_title)
        continue
    seed_asin = matches.iloc[0]["parent_asin"]
    seed_idx  = asin_to_idx.get(seed_asin)
    if seed_idx is None:
        log.info("  '%s': not in embedded set — skipping", seed_title)
        continue
    distances, indices = query_index(
        index, all_embeddings[seed_idx : seed_idx + 1], k=SPOT_CHECK_K + 1
    )
    log.info("  Seed: '%s' (%s)", asin_to_title.get(seed_asin, seed_asin), seed_asin)
    for rank, (dist, neighbour_idx) in enumerate(zip(distances[0], indices[0])):
        if neighbour_idx == seed_idx:
            continue
        neighbour_asin  = all_asins[int(neighbour_idx)]
        neighbour_title = asin_to_title.get(neighbour_asin, "(unknown)")
        log.info("    %d. %s  [L2=%.4f]", rank, neighbour_title, dist)

In [0]:
tier_labels = {
    1: "Full (title+genres+desc+review)",
    2: "Good (title+genres+desc)",
    3: "Thin (title+genres)",
    4: "Bridge (TMDB)",
}
index_size_mb        = os.path.getsize(FAISS_INDEX_PATH) / (1024 ** 2)
embedding_time_label = f"{total_time:.1f} s" if total_time else "N/A (loaded from checkpoint)"

log.info("=" * 60)
log.info("JOB 1 SUMMARY")
log.info("=" * 60)
for t, count in meta["embedding_tier"].value_counts().sort_index().items():
    log.info("  Tier %d — %-35s : %6d (%.1f%%)", t, tier_labels.get(t, ""), count, 100 * count / len(meta))
log.info("Total items embedded : %d", index.ntotal)
log.info("True gaps (skipped)  : %d", true_gaps.sum())
log.info("Embedding time       : %s", embedding_time_label)
log.info("FAISS index size     : %.1f MB", index_size_mb)
log.info("=" * 60)
print("Job 1 complete. Proceed to Job 2 (SVD training).")

dbutils.notebook.exit("SUCCESS")